# Car Dekho — End-to-End Data Analysis
### From Data Exploration and Cleaning to Business Insights

**Objective:** Analyze the Car Dekho used-car dataset to understand resale-price patterns and identify the vehicle, usage and market factors associated with selling price.

**Workflow**
1. Business understanding
2. Data loading
3. Data exploration / profiling
4. Data quality checks
5. Data cleaning and feature engineering
6. Univariate analysis
7. Bivariate / multivariate analysis
8. Correlation and driver analysis
9. Business insights
10. Final conclusions and recommendations


## 1. Business Understanding

### Problem Statement
Used-car resale prices depend on several factors such as original price, vehicle age, kilometers driven, fuel type, transmission, seller type and ownership history.

### Analytical Questions
- What does the dataset contain?
- Is the data complete and consistent?
- Which variables are most strongly associated with selling price?
- How does vehicle age affect resale value?
- How do mileage, fuel type, transmission and seller type influence price?
- Which vehicle segments/models command higher resale prices?
- What actionable conclusions can a buyer, seller or dealer take from the analysis?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('Car Market Trends Analysis with Car Dekho Data.csv')

print('Dataset loaded successfully')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])


## 2. Initial Data Exploration

In [ ]:
# First five records
display(df.head())

# Last five records
display(df.tail())

# Random sample
display(df.sample(5, random_state=42))


In [ ]:
# Dataset dimensions
print('Shape:', df.shape)

# Column names
print('\nColumns:')
for col in df.columns:
    print('-', col)

# Data types
print('\nData types:')
display(df.dtypes.to_frame('dtype'))


In [ ]:
# Statistical summary
display(df.describe().T)

# Categorical summary
display(df.describe(include='object').T)


## 3. Data Quality / Data Cleaning

In [ ]:
# Missing values
missing = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_Percentage': df.isna().mean() * 100
}).sort_values('Missing_Count', ascending=False)

display(missing)

# Duplicate rows
print('Duplicate rows:', df.duplicated().sum())


In [ ]:
# Unique values and potential data-entry issues
quality = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': [df[c].dtype for c in df.columns],
    'Unique_Values': [df[c].nunique() for c in df.columns],
    'Missing_Values': [df[c].isna().sum() for c in df.columns]
})

display(quality)


In [ ]:
# Category inspection
for col in ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']:
    print(f'\n{col}:')
    print(df[col].value_counts(dropna=False))


In [ ]:
# Numeric sanity checks
numeric_cols = ['Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Owner']

for col in numeric_cols:
    print(f'{col}: min={df[col].min()}, max={df[col].max()}')


In [ ]:
# Cleaning
clean_df = df.copy()

# Standardize text columns
text_cols = ['Car_Name', 'Fuel_Type', 'Seller_Type', 'Transmission']
for col in text_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Remove exact duplicate records
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print('Duplicates removed:', before - len(clean_df))

# Validate non-negative fields
for col in ['Selling_Price', 'Present_Price', 'Kms_Driven', 'Owner']:
    print(f'{col} negative values:', (clean_df[col] < 0).sum())

# Validate categorical domains
print('\nFuel types:', clean_df['Fuel_Type'].unique())
print('Seller types:', clean_df['Seller_Type'].unique())
print('Transmission types:', clean_df['Transmission'].unique())
print('Owner values:', sorted(clean_df['Owner'].unique()))


### Cleaning Decision

The supplied dataset is already structurally clean:
- No missing values are present.
- No exact duplicate rows are present.
- Numeric fields contain no negative values.
- Categorical fields contain a small, consistent set of categories.

Therefore, **no imputation is required**. Cleaning focuses on validation, text standardization and creating analysis-ready features.


## 4. Feature Engineering

In [ ]:
# Reference year can be changed if a different business snapshot is required.
REFERENCE_YEAR = clean_df['Year'].max()

clean_df['Car_Age'] = REFERENCE_YEAR - clean_df['Year']
clean_df['Depreciation_Amount'] = clean_df['Present_Price'] - clean_df['Selling_Price']
clean_df['Resale_Retention_%'] = (clean_df['Selling_Price'] / clean_df['Present_Price']) * 100
clean_df['Price_Gap_%'] = (clean_df['Depreciation_Amount'] / clean_df['Present_Price']) * 100

display(clean_df.head())


In [ ]:
# Age bands for business interpretation
clean_df['Age_Band'] = pd.cut(
    clean_df['Car_Age'],
    bins=[-1, 2, 5, 8, 12, np.inf],
    labels=['0-2 years', '3-5 years', '6-8 years', '9-12 years', '13+ years']
)

# Mileage bands using quartiles
clean_df['Mileage_Band'] = pd.qcut(
    clean_df['Kms_Driven'],
    q=4,
    labels=['Low', 'Medium-Low', 'Medium-High', 'High']
)

display(clean_df[['Year','Car_Age','Kms_Driven','Age_Band','Mileage_Band',
                  'Depreciation_Amount','Resale_Retention_%']].head())


## 5. Univariate Analysis

In [ ]:
# Distribution of key numerical variables
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, col in zip(axes.ravel(), ['Selling_Price','Present_Price','Kms_Driven','Year','Car_Age','Resale_Retention_%']):
    sns.histplot(clean_df[col], kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()


In [ ]:
# Categorical distributions
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.ravel(), ['Fuel_Type','Seller_Type','Transmission','Owner']):
    order = clean_df[col].value_counts().index
    sns.countplot(data=clean_df, x=col, order=order, ax=ax)
    ax.set_title(f'{col} Distribution')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 6. Selling Price Analysis

In [ ]:
print('Selling Price Summary')
display(clean_df['Selling_Price'].describe().to_frame().T)

print('\nTop 10 individual listings by selling price')
display(clean_df.nlargest(10, 'Selling_Price')[
    ['Car_Name','Year','Selling_Price','Present_Price','Kms_Driven',
     'Fuel_Type','Seller_Type','Transmission','Owner']
])


In [ ]:
# Average selling price by categorical factors
for col in ['Fuel_Type','Seller_Type','Transmission','Owner','Age_Band','Mileage_Band']:
    result = clean_df.groupby(col, observed=False)['Selling_Price'].agg(['count','mean','median']).sort_values('mean', ascending=False)
    print(f'\n--- Selling Price by {col} ---')
    display(result)


## 7. Vehicle Age vs Selling Price

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=clean_df, x='Car_Age', y='Selling_Price', hue='Fuel_Type', alpha=0.7)
sns.regplot(data=clean_df, x='Car_Age', y='Selling_Price', scatter=False, color='black')
plt.title('Vehicle Age vs Selling Price')
plt.xlabel('Car Age (years)')
plt.ylabel('Selling Price (₹ lakh)')
plt.show()

display(
    clean_df.groupby('Age_Band', observed=False)['Selling_Price']
    .agg(['count','mean','median'])
    .sort_index()
)


## 8. Present Price vs Selling Price

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(data=clean_df, x='Present_Price', y='Selling_Price', hue='Fuel_Type', alpha=0.7)
sns.regplot(data=clean_df, x='Present_Price', y='Selling_Price', scatter=False, color='black')
plt.title('Present Price vs Selling Price')
plt.xlabel('Present Price (₹ lakh)')
plt.ylabel('Selling Price (₹ lakh)')
plt.show()

corr_present = clean_df['Present_Price'].corr(clean_df['Selling_Price'])
print(f'Correlation: {corr_present:.3f}')


## 9. Kilometers Driven vs Selling Price

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(data=clean_df, x='Kms_Driven', y='Selling_Price', hue='Fuel_Type', alpha=0.7)
plt.title('Kilometers Driven vs Selling Price')
plt.xlabel('Kilometers Driven')
plt.ylabel('Selling Price (₹ lakh)')
plt.show()

print('Correlation:', clean_df['Kms_Driven'].corr(clean_df['Selling_Price']))


## 10. Fuel Type, Seller Type and Transmission

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.boxplot(data=clean_df, x='Fuel_Type', y='Selling_Price', ax=axes[0])
axes[0].set_title('Selling Price by Fuel Type')

sns.boxplot(data=clean_df, x='Seller_Type', y='Selling_Price', ax=axes[1])
axes[1].set_title('Selling Price by Seller Type')

sns.boxplot(data=clean_df, x='Transmission', y='Selling_Price', ax=axes[2])
axes[2].set_title('Selling Price by Transmission')

plt.tight_layout()
plt.show()


## 11. Top Models

In [ ]:
model_summary = clean_df.groupby('Car_Name').agg(
    Listings=('Car_Name','size'),
    Avg_Selling_Price=('Selling_Price','mean'),
    Median_Selling_Price=('Selling_Price','median'),
    Avg_Present_Price=('Present_Price','mean'),
    Avg_Kms=('Kms_Driven','mean')
).sort_values('Avg_Selling_Price', ascending=False)

print('Top 10 models by average selling price:')
display(model_summary.head(10))


In [ ]:
# Models with at least 3 listings: more stable comparison
stable_models = model_summary[model_summary['Listings'] >= 3].sort_values(
    'Avg_Selling_Price', ascending=False
)

display(stable_models.head(10))


## 12. Correlation and Driver Analysis

In [ ]:
numeric_analysis = clean_df[
    ['Selling_Price','Present_Price','Kms_Driven','Year','Owner','Car_Age',
     'Depreciation_Amount','Resale_Retention_%']
]

corr = numeric_analysis.corr()

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

print('Correlation with Selling Price:')
display(corr['Selling_Price'].sort_values(ascending=False).to_frame('Correlation'))


## 13. Depreciation / Resale Retention Analysis

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=clean_df, x='Car_Age', y='Resale_Retention_%', hue='Fuel_Type', alpha=0.7)
plt.title('Resale Value Retention vs Vehicle Age')
plt.xlabel('Car Age (years)')
plt.ylabel('Selling Price as % of Present Price')
plt.show()

display(
    clean_df.groupby('Age_Band', observed=False)['Resale_Retention_%']
    .agg(['count','mean','median'])
)


## 14. Outlier Investigation

In [ ]:
# IQR-based outlier counts for major numeric variables
def iqr_outlier_count(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum()), lower, upper

outlier_report = []
for col in ['Selling_Price','Present_Price','Kms_Driven']:
    count, lower, upper = iqr_outlier_count(clean_df[col])
    outlier_report.append([col, count, lower, upper])

display(pd.DataFrame(outlier_report, columns=['Variable','Outlier_Count','Lower_Bound','Upper_Bound']))


### Outlier Decision

Outliers are **identified but not automatically deleted**. High-value vehicles and high-mileage vehicles can be legitimate observations in a used-car market. Removing them without a business reason could distort the market picture.

The analysis therefore retains valid observations and uses robust summaries such as median alongside mean.


## 15. Executive Insights

In [ ]:
# Automatically generate a compact insight dashboard
stats = {}

stats['records'] = len(clean_df)
stats['avg_sell'] = clean_df['Selling_Price'].mean()
stats['median_sell'] = clean_df['Selling_Price'].median()
stats['avg_present'] = clean_df['Present_Price'].mean()
stats['avg_kms'] = clean_df['Kms_Driven'].mean()
stats['present_corr'] = clean_df['Present_Price'].corr(clean_df['Selling_Price'])
stats['kms_corr'] = clean_df['Kms_Driven'].corr(clean_df['Selling_Price'])
stats['age_corr'] = clean_df['Car_Age'].corr(clean_df['Selling_Price'])
stats['fuel_mode'] = clean_df['Fuel_Type'].mode()[0]
stats['trans_mode'] = clean_df['Transmission'].mode()[0]
stats['seller_mode'] = clean_df['Seller_Type'].mode()[0]

print(f"1. Dataset contains {stats['records']} vehicle listings.")
print(f"2. Average selling price is ₹{stats['avg_sell']:.2f} lakh; median is ₹{stats['median_sell']:.2f} lakh.")
print(f"3. Present price has correlation {stats['present_corr']:.2f} with selling price.")
print(f"4. Kilometers driven has correlation {stats['kms_corr']:.2f} with selling price.")
print(f"5. Vehicle age has correlation {stats['age_corr']:.2f} with selling price.")
print(f"6. Most common fuel type: {stats['fuel_mode']}.")
print(f"7. Most common transmission: {stats['trans_mode']}.")
print(f"8. Most common seller type: {stats['seller_mode']}.")


In [ ]:
# Business-oriented comparisons
print('Highest average selling price by fuel:')
display(clean_df.groupby('Fuel_Type')['Selling_Price'].mean().sort_values(ascending=False).to_frame())

print('Highest average selling price by transmission:')
display(clean_df.groupby('Transmission')['Selling_Price'].mean().sort_values(ascending=False).to_frame())

print('Highest average selling price by seller type:')
display(clean_df.groupby('Seller_Type')['Selling_Price'].mean().sort_values(ascending=False).to_frame())

print('Age-band pricing:')
display(clean_df.groupby('Age_Band', observed=False)['Selling_Price'].mean().to_frame())


## 16. Final Conclusion

### Key Takeaways
- The analysis begins with a complete profile of the supplied data and validates its quality before modeling or visualization.
- No missing-value imputation is necessary because the supplied dataset has no missing values.
- Vehicle age, original/present price, kilometers driven and categorical vehicle characteristics are important dimensions for explaining resale-price differences.
- **Present Price is expected to be the strongest direct numerical pricing indicator** in this dataset because it represents the vehicle's original/current market-price basis.
- Age and usage provide important depreciation context: older and more heavily driven cars generally require a larger resale discount, although individual models can behave differently.
- Fuel type, transmission, seller type and ownership should be considered as segmentation variables rather than isolated pricing rules.
- Outliers are retained unless there is evidence that they are data errors, because unusually expensive or high-mileage cars may be genuine market observations.

### Business Recommendations
**For sellers/dealers:** benchmark against comparable model, age, mileage and configuration rather than using one generic price.

**For buyers:** compare selling price with present price, age and kilometers driven to identify whether a listing appears expensive relative to its characteristics.

**For analysts:** a future predictive model can use these cleaned and engineered features to estimate selling price.

### Next Step
Build a supervised machine-learning model such as Linear Regression, Random Forest or Gradient Boosting and evaluate it using MAE, RMSE and R².
